# Evalutaion of Image Quality of CMF vs. VAE

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary
from torchdiffeq import odeint

import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras
from keras import layers, ops, Model, random, models

import numpy as np
import matplotlib.pyplot as plt
from umap import UMAP
import re
import h5py

# if nvidia gpu available use it
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("Nb of Devices: ", torch.cuda.device_count())
    print("Device:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Supported archs:", torch.cuda.get_arch_list())

## 1. Dataset Preperation

In [ ]:
# Load Data
data_path_64 = "/kaggle/input/datasets/mikematician/galaxy/Galaxy10_DECals_64.h5"

with h5py.File(data_path_64, "r") as f:
    images = np.array(f["images"])   # shape (17736, 64, 64, 3)
    labels = np.array(f["ans"])      # shape (17736,)

# Print Shapes
print("Images shape:", images.shape)
print("Labels shape:", labels.shape, "dtype:", labels.dtype)

In [ ]:
# Class names from the Galaxy10 DECaLS documentation
class_names = {
    0: "Disturbed Galaxies",
    1: "Merging Galaxies",
    2: "Round Smooth Galaxies",
    3: "In-between Round Smooth Galaxies",
    4: "Cigar Shaped Smooth Galaxies",  
    5: "Barred Spiral Galaxies",
    6: "Unbarred Tight Spiral Galaxies",
    7: "Unbarred Loose Spiral Galaxies",
    8: "Edge-on Galaxies without Bulge",
    9: "Edge-on Galaxies with Bulge"
}

unique, counts = np.unique(labels, return_counts=True)
print("Class distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u} ({class_names[u]}): {c}")

In [ ]:
# Plot some Images
f, plane = plt.subplots(2, 5, figsize=(24, 8))
for row in plane:
    for axis in row:
        random_index = np.random.randint(0, len(labels))
        axis.imshow(images[random_index])
        axis.set_title(f"class {labels[random_index]}: {class_names[labels[random_index]]}")
plt.show()

## 2. Initialize and load models

### 2.1 CMF

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, groups=32):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.conv1 = nn.Conv2d(in_channels+1, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels+1, out_channels, 3, padding=1)
        self.act   = nn.SiLU()
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def _tc(self, y, t):
        # concatenate t as extra channel
        return torch.cat([y, t.view(-1,1,1,1).expand(y.shape[0],1,y.shape[2],y.shape[3])], dim=1)

    def forward(self, t, y):
        h = self.act(self.norm1(self.conv1(self._tc(y, t))))
        h = self.act(self.norm2(self.conv2(self._tc(h, t))))
        return self.skip(y) + h


class Sampling(nn.Module):
    def __init__(self,  channels, kernelsize_sampling=None, kernelsize_conv=None, upscale=None, upsampling=False):
        super().__init__()
        if not upsampling:
            self.sampling = nn.AvgPool2d(kernel_size=kernelsize_sampling)
            self.conv = nn.Identity()
        else:
            self.sampling = nn.Upsample(scale_factor=upscale, mode="bilinear", align_corners=False)
            self.conv = nn.Conv2d(channels, channels, kernel_size=kernelsize_conv, padding=1)

    def forward(self, z):
        z = self.sampling(z)
        return self.conv(z)


class CNFResBlock(nn.Module):
    def __init__(self, channel, nb_filters, kernelsize):
        super().__init__()
        # ResNet layers
        self.resblock1 = ResBlock(channel, nb_filters)
        self.downsampling1 = Sampling(nb_filters, 2)
        self.resblock2 = ResBlock(nb_filters, 2*nb_filters)
        self.downsampling2 = Sampling(2*nb_filters, 2)
        self.resblock3 = ResBlock(2*nb_filters, 4*nb_filters)
        self.upsampling1 = Sampling(4*nb_filters, kernelsize_conv=kernelsize, upscale=2, upsampling=True)
        self.resblock4 = ResBlock(6*nb_filters, 2*nb_filters)
        self.upsampling2 = Sampling(2*nb_filters, kernelsize_conv=kernelsize, upscale=2, upsampling=True)
        self.resblock5 = ResBlock(3*nb_filters, nb_filters)
        self.outconv = nn.Conv2d(in_channels=nb_filters, out_channels=channel, kernel_size=kernelsize, stride=1, padding=1)

    def forward(self, t, xt):
        y = xt
        # ResNet
        dy_dt = self.resblock1(t, y)
        skip1 = dy_dt
        dy_dt = self.downsampling1(dy_dt)
            
        dy_dt = self.resblock2(t, dy_dt)
        skip2 = dy_dt
        dy_dt = self.downsampling2(dy_dt)
            
        dy_dt = self.resblock3(t, dy_dt)
        dy_dt = self.upsampling1(dy_dt)
        dy_dt = torch.cat([dy_dt, skip2], dim=1) 
            
        dy_dt = self.resblock4(t, dy_dt)
        dy_dt = self.upsampling2(dy_dt)
        dy_dt = torch.cat([dy_dt, skip1], dim=1) 

        dy_dt = self.resblock5(t, dy_dt)
        dy_dt = self.outconv(dy_dt)

        return dy_dt

In [ ]:
# model parameters
data_channels = 3
start_channels = 64

# Initialize model
model = CNFResBlock(data_channels, start_channels, 3).to(device)

In [ ]:
# load model
model.load_state_dict(torch.load("/kaggle/input/models/mikematician/u-net-cnf/pytorch/default/1/flowmatching_checkpoint_epoch50.pt", map_location=device))

### 2.2 VAE

## 3. Generate samples

In [ ]:
nb_samples = 500

### 3.1 CMF samples

In [ ]:
# Sample Function
@torch.no_grad()
def sample_CMF(model, num_samples, t_0, t_end, ode_solver, device, rtol, atol, shape=(3, 64, 64)):
    model.eval()
    y_0 = torch.randn(num_samples, *shape, device=device)
    y = odeint(model, y_0, torch.tensor([t_0, t_end], dtype=torch.float32, device=device), rtol=rtol, atol=atol, method=ode_solver)
    y_0, y_1 = y[:,...]
    
    return y_1, y_0

In [ ]:
t_0 = 0
t_1 = 1
ode_solver = "dopri5"
atol = 1e-7
rtol = 1e-7

generated_images_CMF , gaussian_noise_CMF = sample_CMF(model, nb_samples, t_0, t_1, ode_solver, device, rtol=rtol, atol=atol)

print(generated_images_CMF.shape)

In [ ]:
# convert to cpu
generated_images_cpu = generated_images.cpu()
gaussian_noise_cpu = gaussian_noise.cpu()

In [ ]:
# change dimensions for imshow
plot_gen   = ((generated_images_cpu + 1.0) / 2.0).clamp(0, 1).permute(0, 2, 3, 1)
plot_noise = gaussian_noise_cpu.permute(0, 2, 3, 1)

In [ ]:
# plot example images
rows = 8
nb_plots = rows * 4

fig, ax = plt.subplots(rows, 4, figsize=(4 * 2.2, rows * 2.2))
ax = ax.flatten()
for i in range(1, nb_plots):
    random_nb = np.random.randint(nb_samples)
    ax[i].imshow(plot_gen[random_nb].squeeze())
    ax[i].axis('off')
ax[1].set_title(f"generated")
ax[0].imshow(plot_noise[random_nb].squeeze())
ax[0].set_title(f"gaussian noise")
ax[0].axis('off')

### 3.2 VAE samples